# 02 — Bi-LSTM Baseline

Train and inspect the Bi-LSTM baseline model interactively.
For a full training run, prefer the script:
```
python -m src.training.train_baseline
```

In [ ]:
import os, sys
sys.path.append(os.path.abspath('..'))
import numpy as np
import pandas as pd
import tensorflow as tf

from config import cfg
from src.models.bilstm import Vocabulary, build_bilstm_model, encode_aspect_sentiment_example

np.random.seed(cfg.RANDOM_SEED)
tf.random.set_seed(cfg.RANDOM_SEED)


In [ ]:
train_path = cfg.SENTIMENT_TRAIN_FILE
if not os.path.exists(train_path):
    raise FileNotFoundError(f"{train_path} not found. Run `python -m src.data.prepare_data` first.")

train_df = pd.read_csv(train_path).dropna(subset=['text', 'aspect', 'sentiment'])
val_path = cfg.SENTIMENT_VAL_FILE
val_df = pd.read_csv(val_path).dropna(subset=['text', 'aspect', 'sentiment']) if os.path.exists(val_path) else pd.DataFrame()
print(f"Train: {len(train_df)}  Val: {len(val_df)}")
train_df.head()


In [ ]:
vocab = Vocabulary(max_size=cfg.VOCAB_SIZE)
vocab.build([f"{row.aspect} {row.text}" for row in train_df.itertuples()])
print(f"Vocabulary size: {len(vocab)}")

X_train = np.array([encode_aspect_sentiment_example(vocab, r.text, r.aspect, cfg.MAX_SEQ_LENGTH) for r in train_df.itertuples()])
y_train = train_df['sentiment'].map(cfg.SENTIMENT_TO_ID).values

if not val_df.empty:
    X_val = np.array([encode_aspect_sentiment_example(vocab, r.text, r.aspect, cfg.MAX_SEQ_LENGTH) for r in val_df.itertuples()])
    y_val = val_df['sentiment'].map(cfg.SENTIMENT_TO_ID).values
else:
    X_val, y_val = None, None


In [ ]:
model = build_bilstm_model(vocab_size=len(vocab))
model.summary()


In [ ]:
epochs = 2 if cfg.DEBUG_MODE else cfg.BILSTM_NUM_EPOCHS
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val) if X_val is not None else None,
    epochs=epochs,
    batch_size=cfg.BATCH_SIZE,
    verbose=1,
)


## Plot training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history.get('loss', []), label='train_loss')
if 'val_loss' in history.history:
    axes[0].plot(history.history['val_loss'], label='val_loss')
axes[0].legend(); axes[0].set_title('Loss')

axes[1].plot(history.history.get('accuracy', []), label='train_acc')
if 'val_accuracy' in history.history:
    axes[1].plot(history.history['val_accuracy'], label='val_acc')
axes[1].legend(); axes[1].set_title('Accuracy')
plt.show()
